# 短期记忆
短期记忆是三者的组合：
State（会话内部状态）+ Checkpointer（持久化机制）+ Thread ID（会话作用域）
- State ：默认 存储历史消息列表messages ，通过State 管理历史消息
  
- Checkpointer ：负责将State 作为检查点持久化保存，检查点是某个时刻的State 快照
  
- Thread ID ：用于唯一标识State ，LangChain运行时会按照 thread_id 读写State快照

## 短期记忆的实现
短期记忆的实现，需要使用到`State`和`Checkpointer`。

## 基于内存的持久化器
最便捷的使用方式，适合快速测试或调试

In [ ]:
# 基于装饰器实现
# 1、模型的初始化
import os
from dotenv import load_dotenv
from langchain_qwq import ChatQwen
from rich import print as rprint

custom_profile = {
"max_input_tokens": 128_000 # 最大上下文长度
}

# 从.env文件中加载环境变量
load_dotenv(override=True)
# 模型的初始化
model = ChatQwen(
    model="qwen3.6-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
    profile=custom_profile, # 手动添加的配置项
)

In [4]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from rich import print as rprint

checkpointer = InMemorySaver() # 创建了内存级的记忆存储

# 1. 创建 Agent 时添加 checkpointer
agent = create_agent(
    model=model,
    checkpointer=checkpointer,  # 添加内存管理，让agent具备了存储的能力
)

# 2. 调用时指定 thread_id，同一个thread_id的对话共享同一个记忆
config = {
    "configurable": {
        "thread_id": "1",
    }
}

print("\n第一轮对话：")
response1 = agent.invoke(
    {"messages": [HumanMessage("我叫张三")]},
    config=config,  # 传入 config
)
print(f"Agent: {response1['messages'][-1].content}")

print("\n第二轮对话：")
response2 = agent.invoke(
    {"messages": [HumanMessage("我叫什么？")]},
    config=config,  # 使用相同的 thread_id
)
print(f"Agent: {response2['messages'][-1].content}")

latest_state = agent.get_state(config)
rprint(latest_state)


第一轮对话：
Agent: 你好，张三！很高兴认识你。今天有什么我可以帮你的吗？😊

第二轮对话：
Agent: 你叫**张三**。很高兴认识你！今天有什么我可以帮你的吗？😊


StateSnapshot(
    values={
        'messages': [
            HumanMessage(
                content='我叫张三',
                additional_kwargs={},
                response_metadata={},
                id='64f5ffe0-78ea-483c-a3a9-6a63d1789f72'
            ),
            AIMessage(
                content='你好，张三！很高兴认识你。今天有什么我可以帮你的吗？😊',
                additional_kwargs={
                    'refusal': None,
                    'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User 
says: "我叫张三" (My name is Zhang San)\n   - This is a simple self-introduction in Chinese.\n   - "张三" is a 
common placeholder name in Chinese, similar to "John Doe" in English, but it can also be a real name.\n\n2.  
**Identify Key Elements:**\n   - Language: Chinese\n   - Content: Personal introduction (name)\n   - Intent: Likely
starting a conversation, testing the system, or just making a statement\n\n3.  **Determine Appropriate 
Response:**\n   - Acknowledge the greeting/introduction politely\n   - Use the user\'s name appropriately\n   - 
Keep it friendly and open-ended to encourage further conversation\n   - Match the language (Chinese)\n\n4.  **Draft
Response (Mental Refinement in Chinese):**\n   - 你好，张三！很高兴认识你。有什么我可以帮你的吗？\n   - (Hello, 
Zhang San! Nice to meet you. How can I help you today?)\n\n5.  **Check Against Guidelines:**\n   - Polite? Yes\n   
- Uses name? Yes\n   - Open-ended? Yes\n   - Language matches? Yes\n   - No overcomplication? Yes\n\n6.  **Final 
Output Generation:** (matches the drafted response)\n   - 你好，张三！很高兴认识你。今天有什么我可以帮你的吗？😊 \n
- (Added a slight variation for natural flow and warmth)✅'
                },
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 346,
                        'prompt_tokens': 12,
                        'total_tokens': 358,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 325,
                            'rejected_prediction_tokens': None,
                            'text_tokens': 346
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 12}
                    },
                    'model_provider': 'dashscope',
                    'model_name': 'qwen3.6-flash',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-9fdd4320-ca76-9898-b2b0-edde897b8674',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019f4233-c819-78c0-a246-277d93c7fcd5-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 12,
                    'output_tokens': 346,
                    'total_tokens': 358,
                    'input_token_details': {},
                    'output_token_details': {'reasoning': 325}
                }
            ),
            HumanMessage(
                content='我叫什么？',
                additional_kwargs={},
                response_metadata={},
                id='32a26fd7-75cc-4ce8-b85e-c8cdec7b7f8e'
            ),
            AIMessage(
                content='你叫**张三**。很高兴认识你！今天有什么我可以帮你的吗？😊',
                additional_kwargs={
                    'refusal': None,
                    'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User 
says: "我叫什么？" (What is my name?)\n   - Previous context: User said "我叫张三" (My name is Zhang San).\n\n2.  
**Identify Key Information:**\n   - The user explicitly stated their name in the first turn: "张三" (Zhang San).\n 
- The question is a straightforward recall test based on the conversation history.\n\n3.  **Formulate Response:**\n
- Acknowledge the name clearly.\n   - Keep 

In [ ]:
# 更新线程ID，会重新开启对话 
config2 = {
    "configurable": {
        "thread_id": "2"
    }
}
response4 = agent.invoke(
    {"messages": [HumanMessage("你还记得我叫什么名字么？")]},
    config=config2
)
print(response4['messages'][-1].content)

抱歉，作为一个人工智能，我无法记住或存储用户的个人信息（包括姓名）。出于隐私保护的设计，每次对话对我来说都是全新的，我不会保留历史记录或个人数据。

不过，如果你愿意在现在的对话中告诉我你的名字，我很乐意在接下来的交流中用你喜欢的称呼来跟你聊天！😊


## 关键步骤说明
第一步：初始化记忆引擎：checkpointer = InMemorySaver()，创建一个内存级的记忆存储。
> InMemorySaver内存中保存，进程结束就丢失数据，适合测试。生产环境可换成数据库持久化的 SqliteSaver 、 PostgresSaver 等

第二步：绑定Agent：在create_agent时传入checkpointer，让agent具备了存储的能力

第三步：设定会话ID：通过config = {"configurable": {"thread_id": "1"}}指定线程标识，同一个thread_id共享记忆，不同thread_id完全隔离。